# Persian STT: audio preprocessing comparison (Whisper large-v3)

This notebook compares six audio preprocessing configurations for `openai/whisper-large-v3` on the project's own recordings: the same 112 samples used in `stt-model-comparison.ipynb`, where Whisper large-v3 was selected as the STT model. The metric is corpus-level WER after text normalization.

**Pipeline**
1. Model: `openai/whisper-large-v3`, used as-is (no fine-tuning).
2. Data: `voice_dataset/audio` with reference transcripts from `metadata.xlsx`.
3. Preprocessing variants: raw (resample + mono), denoise, loudness normalization, silence trimming, and combinations of these.
4. Per-sample error analysis (substitutions, deletions, insertions) for one selected configuration.

A first run of this comparison on Common Voice (`veziriii/common-voice-17`) ranked the configurations differently; the runs here use the project's own recordings. See the thesis (section 3-4) for the discussion.

**To run**
- Set `DATASET_ROOT` to the location of `voice_dataset`. The recordings are private and not included in the repository.

## 1. Install dependencies

In [ ]:
!pip install -q -U "numpy<2.2" transformers accelerate jiwer soundfile librosa hazm noisereduce pyloudnorm


## 2. Imports and configuration

In [1]:
import os
import gc
import re
import time
import glob
import tempfile

import torch
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr
import pyloudnorm as pyln

from transformers import WhisperProcessor, WhisperForConditionalGeneration, pipeline
from jiwer import wer
from hazm import Normalizer

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
print("Device:", device)

MODEL_ID = "openai/whisper-large-v3"

# Root of the dataset (same samples as stt-model-comparison.ipynb):
#   .../voice_dataset/audio/          -> audio files
#   .../voice_dataset/metadata.xlsx   -> Sheet1 with columns file_name, emotion, transcript
DATASET_ROOT = "/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset"
AUDIO_DIR = os.path.join(DATASET_ROOT, "audio")
METADATA_XLSX = os.path.join(DATASET_ROOT, "metadata.xlsx")
METADATA_SHEET = "Sheet1"

TARGET_SR = 16000
normalizer = Normalizer()


Device: cuda


## 3. Load the dataset (audio + transcript pairs)

Audio files are matched to the reference transcripts in `metadata.xlsx` with the same matching logic as `stt-model-comparison.ipynb`, so the two experiments are directly comparable.

In [2]:
AUDIO_EXTS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

def find_audio_path(file_name: str):
    base = str(file_name).strip()
    direct = os.path.join(AUDIO_DIR, base)
    if os.path.exists(direct):
        return direct
    for ext in AUDIO_EXTS:
        candidate = os.path.join(AUDIO_DIR, base + ext)
        if os.path.exists(candidate):
            return candidate
    matches = glob.glob(os.path.join(AUDIO_DIR, "**", base + ".*"), recursive=True)
    return matches[0] if matches else None


def load_audio_robust(path: str, sr: int = TARGET_SR):
    """Try librosa/soundfile first; fall back to ffmpeg conversion for formats
    like m4a that librosa/soundfile may not read directly."""
    try:
        array, _ = librosa.load(path, sr=sr, mono=True)
        return array
    except Exception:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp_wav = tmp.name
        try:
            result = os.system(f'ffmpeg -y -loglevel error -i "{path}" -ar {sr} -ac 1 "{tmp_wav}"')
            if result != 0:
                raise RuntimeError(f"ffmpeg failed to convert {path}")
            array, _ = librosa.load(tmp_wav, sr=sr, mono=True)
            return array
        finally:
            if os.path.exists(tmp_wav):
                os.remove(tmp_wav)


meta_df = pd.read_excel(METADATA_XLSX, sheet_name=METADATA_SHEET)
print(f"Loaded metadata with {len(meta_df)} rows.")

manifest_rows = []
missing = []
for _, row in meta_df.iterrows():
    audio_path = find_audio_path(row["file_name"])
    if audio_path is None:
        missing.append(row["file_name"])
        continue
    manifest_rows.append({"audio_path": audio_path, "transcript": row["transcript"]})

if missing:
    print(f"WARNING: {len(missing)} audio file(s) not found (sample): {missing[:5]}")

manifest_df = pd.DataFrame(manifest_rows)
print(f"Manifest ready with {len(manifest_df)} matched samples.")

test_subset = []
failed_audio = []
for _, row in manifest_df.iterrows():
    try:
        array = load_audio_robust(row["audio_path"], sr=TARGET_SR)
    except Exception as e:
        failed_audio.append((row["audio_path"], str(e)))
        continue
    test_subset.append({
        "audio": {"array": array, "sampling_rate": TARGET_SR},
        "transcript": str(row["transcript"]),
    })

if failed_audio:
    print(f"WARNING: {len(failed_audio)} audio file(s) could not be read (sample): {failed_audio[:3]}")
print(f"Test samples selected for evaluation: {len(test_subset)}")


Loaded metadata with 112 rows.
Manifest ready with 112 matched samples.
Test samples selected for evaluation: 112


## 4. Shared helper functions (text normalization, WER, preprocessing, model loading)

In [3]:
def clean_text(text: str) -> str:
    """Normalize Persian text before WER computation."""
    if not isinstance(text, str):
        return ""
    text = normalizer.normalize(text)
    text = re.sub(r"[\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)  # remove diacritics
    text = re.sub(r"[^\w\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_wer(references, hypotheses):
    """Compute WER over reference/hypothesis pairs, skipping empty ones."""
    refs_clean = [clean_text(r) for r in references]
    hyps_clean = [clean_text(h) for h in hypotheses]
    pairs = [(r, h) for r, h in zip(refs_clean, hyps_clean) if len(r) > 0 and len(h) > 0]
    if not pairs:
        return 1.0, 0
    refs_valid = [p[0] for p in pairs]
    hyps_valid = [p[1] for p in pairs]
    return wer(refs_valid, hyps_valid), len(pairs)


def preprocess_audio(audio_array, sr, apply_denoise=False, apply_trim=True, apply_loudness_norm=True):
    """Full audio preprocessing pipeline with toggleable steps."""
    if audio_array.ndim > 1:
        audio_array = librosa.to_mono(audio_array.T)

    if sr != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    if apply_denoise:
        try:
            audio_array = nr.reduce_noise(y=audio_array, sr=sr, stationary=False, prop_decrease=0.75)
        except Exception as e:
            print(f"Warning: denoise failed ({e})")

    if apply_loudness_norm:
        try:
            meter = pyln.Meter(sr)
            loudness = meter.integrated_loudness(audio_array)
            if np.isfinite(loudness):
                audio_array = pyln.normalize.loudness(audio_array, loudness, -23.0)
        except Exception as e:
            print(f"Warning: loudness normalization failed ({e})")

    max_val = np.max(np.abs(audio_array))
    if max_val > 1.0:
        audio_array = audio_array / max_val * 0.98

    if apply_trim:
        audio_array, _ = librosa.effects.trim(audio_array, top_db=30)

    return audio_array, sr


PREPROCESSING_VARIANTS = {
    "raw (resample+mono only)":            dict(apply_denoise=False, apply_loudness_norm=False, apply_trim=False),
    "denoise only":                        dict(apply_denoise=True,  apply_loudness_norm=False, apply_trim=False),
    "loudness normalization only":         dict(apply_denoise=False, apply_loudness_norm=True,  apply_trim=False),
    "trim silence only":                   dict(apply_denoise=False, apply_loudness_norm=False, apply_trim=True),
    "loudness norm + trim (no denoise)":   dict(apply_denoise=False, apply_loudness_norm=True,  apply_trim=True),
    "denoise + loudness + trim (all)":     dict(apply_denoise=True,  apply_loudness_norm=True,  apply_trim=True),
}


def load_asr_pipeline(model_id):
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True,
    ).to(device)
    asr = pipeline(
        task="automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=0 if device == "cuda" else -1,
        generate_kwargs={"language": "fa", "task": "transcribe"},
        chunk_length_s=30,
    )
    return asr, model, processor


def unload_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_preprocessing_search(asr, dataset_subset, reference_field="transcript"):
    """Test all preprocessing variants on a given ASR pipeline, return comparison DataFrame."""
    references = [s.get(reference_field, "") for s in dataset_subset]
    rows = []
    for variant_name, params in PREPROCESSING_VARIANTS.items():
        hypotheses = []
        start_time = time.time()
        for sample in dataset_subset:
            audio_array = sample["audio"]["array"]
            sr = sample["audio"]["sampling_rate"]
            audio_array, _ = preprocess_audio(audio_array, sr, **params)
            try:
                hypotheses.append(asr(audio_array)["text"])
            except Exception:
                hypotheses.append("")
        elapsed = time.time() - start_time

        variant_wer, n_valid = compute_wer(references, hypotheses)
        accuracy = (1 - variant_wer) * 100
        rows.append({
            "preprocessing": variant_name,
            "wer_percent": round(variant_wer * 100, 2),
            "accuracy_percent": round(accuracy, 2),
            "valid_samples": n_valid,
            "time_seconds": round(elapsed, 1),
        })
        print(f"{variant_name:38s} -> WER: {variant_wer*100:.2f}% | Accuracy: {accuracy:.2f}%")

    return pd.DataFrame(rows).sort_values("wer_percent").reset_index(drop=True)

print("Helper functions ready.")


Helper functions ready.


---
## Stage 1: Preprocessing comparison of `openai/whisper-large-v3` on our own dataset

In [4]:
asr_base, base_model, base_processor = load_asr_pipeline(MODEL_ID)
print(f"Loaded base model: {MODEL_ID}")

baseline_comparison_df = run_preprocessing_search(asr_base, test_subset, reference_field="transcript")
print("\n=== Preprocessing comparison on our own dataset (lower WER is better) ===")
print(baseline_comparison_df.to_string(index=False))


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loaded base model: openai/whisper-large-v3


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


raw (resample+mono only)               -> WER: 35.33% | Accuracy: 64.67%
denoise only                           -> WER: 35.90% | Accuracy: 64.10%
loudness normalization only            -> WER: 35.71% | Accuracy: 64.29%
trim silence only                      -> WER: 44.06% | Accuracy: 55.94%
loudness norm + trim (no denoise)      -> WER: 34.28% | Accuracy: 65.72%


/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


denoise + loudness + trim (all)        -> WER: 36.18% | Accuracy: 63.82%

=== Preprocessing comparison on our own dataset (lower WER is better) ===
                    preprocessing  wer_percent  accuracy_percent  valid_samples  time_seconds
loudness norm + trim (no denoise)        34.28             65.72            112         218.7
         raw (resample+mono only)        35.33             64.67            112         242.6
      loudness normalization only        35.71             64.29            112         243.2
                     denoise only        35.90             64.10            112         226.4
  denoise + loudness + trim (all)        36.18             63.82            112         224.2
                trim silence only        44.06             55.94            112         241.5


### Results (from `stt_preprocessing_comparison_own_data.csv`)

| preprocessing | WER (%) | 100 - WER (%) | time (s) |
|---|---|---|---|
| loudness norm + trim (no denoise) | 34.28 | 65.72 | 218.1 |
| raw (resample + mono only) | 35.33 | 64.67 | 242.3 |
| loudness normalization only | 35.71 | 64.29 | 241.0 |
| denoise only | 35.90 | 64.10 | 226.1 |
| denoise + loudness + trim (all) | 36.18 | 63.82 | 225.4 |
| trim silence only | 44.06 | 55.94 | 240.0 |

All rows use 112 valid samples. `accuracy_percent` in the code is `100 - WER`, not a classification accuracy.

In [5]:
# Stage 1b: per-sample error analysis for one preprocessing configuration.
# Uses the model loaded above (asr_base), so it must run before the unload cell.

import jiwer

# Configuration to analyze. Use "raw (resample+mono only)" to analyze the
# configuration deployed in the backend.
ERROR_ANALYSIS_CONFIG_NAME = "loudness norm + trim (no denoise)"
BEST_CONFIG = PREPROCESSING_VARIANTS[ERROR_ANALYSIS_CONFIG_NAME]


def _word_measures(ref_clean, hyp_clean):
    """Works across jiwer versions (process_words in newer jiwer, compute_measures in older)."""
    try:
        m = jiwer.process_words(ref_clean, hyp_clean)
        return dict(wer=m.wer, substitutions=m.substitutions,
                    deletions=m.deletions, insertions=m.insertions, hits=m.hits)
    except AttributeError:
        m = jiwer.compute_measures(ref_clean, hyp_clean)
        return dict(wer=m["wer"], substitutions=m["substitutions"],
                    deletions=m["deletions"], insertions=m["insertions"], hits=m["hits"])


# manifest_df and test_subset are aligned by position only if every audio file loaded.
assert len(manifest_df) == len(test_subset), (
    "Some audio files failed to load, so manifest_df and test_subset are misaligned."
)

per_sample_rows = []
for idx, sample in enumerate(test_subset):
    audio_array = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    ref = sample["transcript"]

    proc_audio, _ = preprocess_audio(audio_array, sr, **BEST_CONFIG)
    try:
        hyp = asr_base(proc_audio)["text"]
    except Exception as e:
        hyp = ""
        print(f"[warn] ASR failed on sample {idx}: {e}")

    ref_clean = clean_text(ref)
    hyp_clean = clean_text(hyp)
    if len(ref_clean) == 0 or len(hyp_clean) == 0:
        continue  # same skip rule as compute_wer() above

    measures = _word_measures(ref_clean, hyp_clean)

    per_sample_rows.append({
        "index": idx,
        "audio_path": manifest_df.iloc[idx]["audio_path"],
        "reference": ref,
        "hypothesis": hyp,
        "wer_percent": round(measures["wer"] * 100, 2),
        "substitutions": measures["substitutions"],
        "deletions": measures["deletions"],
        "insertions": measures["insertions"],
        "hits": measures["hits"],
        "ref_word_count": len(ref_clean.split()),
    })

error_df = pd.DataFrame(per_sample_rows).sort_values("wer_percent", ascending=False).reset_index(drop=True)

print(f"Per-sample analysis on {len(error_df)} valid samples "
      f"(dropped {len(test_subset) - len(error_df)} empty ref/hyp pairs).")
print(error_df[["wer_percent", "substitutions", "deletions", "insertions", "ref_word_count"]].describe())

error_df.to_csv("/kaggle/working/stt_per_sample_errors.csv", index=False, encoding="utf-8-sig")
print("Saved to /kaggle/working/stt_per_sample_errors.csv")

print("\n=== per-sample results, worst WER first ===")
for _, row in error_df.iterrows():
    print(f"[WER {row['wer_percent']}%] sub={row['substitutions']} del={row['deletions']} ins={row['insertions']} "
          f"({row['audio_path']})")
    print("  REF:", row["reference"])
    print("  HYP:", row["hypothesis"])
    print()

Per-sample analysis on 112 valid samples (dropped 0 empty ref/hyp pairs).
       wer_percent  substitutions   deletions  insertions  ref_word_count
count   112.000000     112.000000  112.000000  112.000000      112.000000
mean     35.600625       2.383929    0.250000    0.589286        9.401786
std      28.602710       1.973909    0.690997    0.844202        3.431334
min       0.000000       0.000000    0.000000    0.000000        3.000000
25%      16.075000       1.000000    0.000000    0.000000        7.000000
50%      30.790000       2.000000    0.000000    0.000000        9.000000
75%      50.832500       3.000000    0.000000    1.000000       11.000000
max     125.000000       9.000000    6.000000    4.000000       21.000000
Saved to /kaggle/working/stt_per_sample_errors.csv

=== per-sample results, worst WER first ===
[WER 125.0%] sub=6 del=0 ins=4 (/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset/audio/speaker_17/054_ANG.m4a)
  REF: بچه انقد 

In [6]:
unload_model(base_model)

## Save results

In [7]:
baseline_comparison_df.to_csv("/kaggle/working/stt_preprocessing_comparison_own_data.csv", index=False)
print("Saved to /kaggle/working/stt_preprocessing_comparison_own_data.csv")


Saved to /kaggle/working/stt_preprocessing_comparison_own_data.csv


## Summary

| Step | Result |
|---|---|
| Dataset | Project's own 112-sample dataset (`voice_dataset`), same as `stt-model-comparison.ipynb` |
| Model | `openai/whisper-large-v3`, no fine-tuning |
| Experiment | 6 preprocessing configurations compared by WER |
| Output | `stt_preprocessing_comparison_own_data.csv`, `stt_per_sample_errors.csv` |